In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [1]:
# ==============================================================================
# CELDA 1: LIBRERÍAS + INGESTA SEP 2018-2019
# ==============================================================================
import pandas as pd

RUTA = RUTA_RAW + 'sep/'

archivos = {
    2018: 'Preferentes_Prioritarios_y_Beneficiarios_SEP_2018.csv',
    2019: 'Preferentes_Prioritarios_y_Beneficiarios_SEP_2019.csv',
}

sep_brutas = {}
for anio, nombre in archivos.items():
    ruta = RUTA + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            sep_brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {sep_brutas[anio].shape[0]} filas x {sep_brutas[anio].shape[1]} columnas")

print("\nColumnas 2018:", sep_brutas[2018].columns.tolist())

Mounted at /content/drive
OK | 2018 | 11524 filas x 23 columnas
OK | 2019 | 11401 filas x 23 columnas

Columnas 2018: ['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'COD_REG_RBD', 'COD_PRO_RBD', 'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD', 'NOM_DEPROV_RBD', 'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'ESTADO_ESTAB', 'CONVENIO_SEP', 'AÑO_INGRESO_SEP', 'CLASIFICACION_SEP', 'EE_GRATUITO', 'N_PRIO', 'N_PRIO_BEN', 'N_PREF', 'N_PREF_BEN', 'N_BEN']


In [2]:
# ==============================================================================
# CELDA 2: TASA DE VULNERABILIDAD SEP POR RBD (bienio 2018-19)
# ==============================================================================
def procesar_sep(df):
    d = df.copy()
    d['RBD'] = pd.to_numeric(d['RBD'], errors='coerce')
    d = d.dropna(subset=['RBD'])
    d['rbd'] = d['RBD'].astype('Int64').astype(str)

    for col in ['N_PRIO', 'N_PREF', 'N_BEN']:
        d[col] = pd.to_numeric(d[col], errors='coerce').fillna(0)

    d['n_vulnerables'] = d['N_PRIO'] + d['N_PREF']  # prioritarios + preferentes
    d['tiene_convenio_sep'] = pd.to_numeric(d['CONVENIO_SEP'], errors='coerce').fillna(0).astype(int)

    return d[['rbd', 'n_vulnerables', 'N_BEN', 'tiene_convenio_sep']].rename(columns={'N_BEN': 'n_beneficiarios_sep'})

sep_limpios = {anio: procesar_sep(df) for anio, df in sep_brutas.items()}

for anio in sep_limpios:
    sep_limpios[anio] = sep_limpios[anio].groupby('rbd', as_index=False).mean()
    print(f"{anio}: {sep_limpios[anio].shape[0]} colegios únicos")

pool = pd.concat(sep_limpios.values(), ignore_index=True)
sep_1819 = pool.groupby('rbd', as_index=False).mean()

print(f"\nBienio 2018-19: {sep_1819.shape[0]} colegios")
print(sep_1819.describe())

RUTA_SALIDA = RUTA_PROCESADOS
sep_1819.to_parquet(RUTA_SALIDA + 'sep_2018_19_por_rbd.parquet', index=False)
print("Guardado OK")

2018: 11524 colegios únicos
2019: 11401 colegios únicos

Bienio 2018-19: 11604 colegios
       n_vulnerables  n_beneficiarios_sep  tiene_convenio_sep
count   11604.000000         11604.000000        11604.000000
mean      222.937134           171.939762            0.689805
std       286.796084           267.957020            0.460890
min         1.000000             0.000000            0.000000
25%        30.500000             0.000000            0.000000
50%       107.000000            40.000000            1.000000
75%       320.500000           254.000000            1.000000
max      3277.500000          3211.500000            1.000000
Guardado OK


In [3]:
# ==============================================================================
# CELDA 3: INTEGRAR SEP A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = RUTA_PROCESADOS
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v8.parquet')
sep = pd.read_parquet(RUTA + 'sep_2018_19_por_rbd.parquet')

df_modelo_v9 = pd.merge(df_modelo, sep, on='rbd', how='left', validate='one_to_one')

print(f"Filas: {len(df_modelo_v9)} (antes: {len(df_modelo)})")
print(f"Con dato SEP: {df_modelo_v9['n_vulnerables'].notna().sum()}")

df_modelo_v9.to_parquet(RUTA + 'tabla_modelo_final_v9.parquet', index=False)
print(df_modelo_v9.shape)

Filas: 7754 (antes: 7754)
Con dato SEP: 7754
(7754, 60)
